# Data Operations Pipeline
This notebook demonstrates how to implement a chain of operations on DataFrames representing EU countries over years.
We use `pandas` DataFrames because they inherently handle row (countries) and column (years) indices perfectly, and support underlying numpy matrix operations natively.
We also introduce an `Operation` class that overloads the `>>` operator to compose functions, acting like a monoid for function composition (a functional delegate pattern).

In [ ]:
import pandas as pd
import numpy as np

countries = ['DE', 'FR', 'IT', 'ES', 'PL']

# Create sample data tables T_i
np.random.seed(42)
years_T1 = list(range(2005, 2025))
T_1 = pd.DataFrame(np.random.rand(len(countries), len(years_T1)), index=countries, columns=years_T1)

years_T2 = [2018, 2019, 2020, 2021, 2022]
T_2 = pd.DataFrame(np.random.rand(len(countries), len(years_T2)), index=countries, columns=years_T2)

cols_T3 = ['y1', 'y2', 'y3', 'y4', 'y5']
T_3 = pd.DataFrame(np.random.rand(len(countries), len(cols_T3)), index=countries, columns=cols_T3)

display(T_1.head(), T_2.head(), T_3.head())

## Operation Composer (Monoid Pattern)
We define an `Operation` class that wraps a function. The `>>` operator is overloaded to compose two operations together. This creates a highly readable, declarative pipeline syntax.

In [ ]:
class Operation:
    def __init__(self, func):
        self.func = func
        
    def __call__(self, x):
        return self.func(x)
        
    def __rshift__(self, other):
        # Monoid-like composition: op1 >> op2
        return Operation(lambda x: other(self.func(x)))

# Factory functions to generate operations with parameters
def select_years(start, end):
    return Operation(lambda df: df.loc[:, start:end])

def select_specific_years(years):
    return Operation(lambda df: df[years])

def select_column(col):
    return Operation(lambda df: df[col])

def weighted_average(weights):
    # Matrix multiplication handles the weighted sum across columns naturally
    # df shape: (N_countries, N_years), weights shape: (N_years,)
    # Pandas @ operator delegates to numpy matrix multiplication
    return Operation(lambda df: df @ weights)

# Parameter-less operations
arithmetic_average = Operation(lambda df: df.mean(axis=1))
apply_sqrt = Operation(lambda s: np.sqrt(s))
apply_log = Operation(lambda s: np.log(s))

## A_1: Weighted average for years 2010-2019

In [ ]:
# Years 2010 to 2019 inclusive -> 10 years
weights_A1 = np.ones(10) / 10  # Example equal weights, could be any vector

# Build the pipeline
pipeline_1 = select_years(2010, 2019) >> weighted_average(weights_A1)

# Execute 
A_1 = pipeline_1(T_1)
display(A_1)

## A_2: Arithmetic average for specific years, then sqrt

In [ ]:
target_years = [2018, 2020, 2022]

pipeline_2 = select_specific_years(target_years) >> arithmetic_average >> apply_sqrt

A_2 = pipeline_2(T_2)
display(A_2)

## A_3: Log of column 'y4'

In [ ]:
pipeline_3 = select_column('y4') >> apply_log

A_3 = pipeline_3(T_3)
display(A_3)

## Task 2: Conditional Selection with Masks
We construct `A_x` using a pipeline that preserves the single-input, single-output Monoid pattern. 
Assuming `A_l, W_l` implies multiplying the array by the weight array (`A_l * W_l`), we can create operations for element-wise `multiply` and boolean `mask` conditional selection using `pd.DataFrame.where`.

In [ ]:
np.random.seed(42)
A_l = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])
A_k = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])
W_l = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])
W_k = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])

M_i = pd.DataFrame(np.random.choice([True, False], size=(5, 3)), index=countries, columns=['y1', 'y2', 'y3'])

# Operations that maintain the 1-to-1 pipeline flow
def multiply_by(weight_df):
    return Operation(lambda df: df * weight_df)

def take_where(mask, fallback_df):
    # keeps original df values where `mask` is True, otherwise takes `fallback_df` values
    return Operation(lambda df: df.where(mask, fallback_df))

# Constructing the pipeline for A_x
# Read as: Take A_l -> Multiply by W_l -> Where M_i is True keep this, else fallback to (A_k * W_k)
build_A_x = multiply_by(W_l) >> take_where(M_i, A_k * W_k)

# Execute pipeline on A_l
A_x = build_A_x(A_l)

display("Mask M_i:", M_i)
display("Result A_x:", A_x)